# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Ranking / Scoring

**Why:**
Content teams manage thousands of pages across a site but only have time to review 20 to 50 per week. Under the hood, this is framed as a **scoring model** that outputs a probability/urgency score between 0.0 and 1.0 for each page representing its likelihood and severity of traffic/value decline. Sorting the dataset by this score produces a **ranked review queue**, ensuring human reviewers spend their limited hours on the highest-risk pages first.

In [1]:
import pandas as pd
import numpy as np

# Demonstrate output structure: Score -> Rank
demo_scores = pd.DataFrame({
    "page_id": ["page_101", "page_102", "page_103", "page_104", "page_105"],
    "decline_probability_score": [0.94, 0.88, 0.72, 0.41, 0.12]
})

demo_scores["review_priority_rank"] = demo_scores["decline_probability_score"].rank(ascending=False).astype(int)
print("Demonstration of Ranked Review Queue Output:")
print(demo_scores)


Demonstration of Ranked Review Queue Output:
    page_id  decline_probability_score  review_priority_rank
0  page_101                       0.94                     1
1  page_102                       0.88                     2
2  page_103                       0.72                     3
3  page_104                       0.41                     4
4  page_105                       0.12                     5


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Real Target vs. Proxy Label:**
- **Real Target:** True underlying content decay that degrades user experience, causes loss of business value, or renders information obsolete.
- **Proxy Label:** `trend_direction == 'down'` (derived from historical page traffic metrics).

**Where the label comes from:**
It is a proxy rule derived from observational performance data. True content decay cannot be directly observed in raw web analytics logs—only traffic changes can. `trend_direction == 'down'` serves as an operational proxy label because traffic drops can occur due to external factors like seasonality or search algorithm volatility even when content quality remains intact.

In [2]:
import pandas as pd

# Load dataset and check distribution of the proxy label
try:
    df = pd.read_csv("content_refresh_anonymized.csv")
except FileNotFoundError:
    df = pd.read_csv("../content_refresh_anonymized.csv")

declining_count = (df['trend_direction'] == 'down').sum()
total_count = len(df)
pct_declining = (declining_count / total_count) * 100

print(f"Total Pages Evaluated: {total_count:,}")
print(f"Declining Pages (Proxy Label = 1): {declining_count:,} ({pct_declining:.1f}%)")


ParserError: Error tokenizing data. C error: Expected 44 fields in line 4682, saw 72


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Metric:** Precision@K (where $K$ matches weekly review capacity, e.g., $K = 20$ or $K = 50$)
**Secondary Metric:** PR-AUC (Precision-Recall Area Under the Curve)

**Justification & Defense:**
Reviewer capacity is strictly constrained. A **False Positive (FP)** wastes valuable analyst hours auditing a healthy page. A **False Negative (FN)** leaves a declining page unreviewed temporarily, but it can be caught in subsequent runs as traffic drops further.

Because analysts inspect a fixed budget of $K$ pages per week, **Precision@K** directly measures business efficiency: what percentage of the top $K$ flagged pages actually required an update. **PR-AUC** evaluates global ranking performance across the dataset without being skewed by overall class balance.


In [ ]:
import numpy as np

def precision_at_k(y_true, y_scores, k=20):
    top_k_indices = np.argsort(y_scores)[::-1][:k]
    return np.mean(np.array(y_true)[top_k_indices])

# Simulated precision check
y_true_sample = np.array([1, 1, 0, 1, 0, 1, 0, 0, 1, 0])
y_scores_sample = np.array([0.95, 0.88, 0.81, 0.75, 0.60, 0.52, 0.40, 0.30, 0.20, 0.10])

p_at_5 = precision_at_k(y_true_sample, y_scores_sample, k=5)
print(f"Precision@5 on sample predictions: {p_at_5:.2f} ({p_at_5 * 100}% of top 5 flagged pages needed review)")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:**
**One row = One unique URL / Page** over a historical evaluation window (e.g., weekly snapshot).

In [ ]:
import pandas as pd

try:
    df = pd.read_csv("content_refresh_anonymized.csv")
except FileNotFoundError:
    df = pd.read_csv("../content_refresh_anonymized.csv")

print(f"Dataset Shape: {df.shape}")
print("\nUnit of Analysis Display (1 Row = 1 Page):")
display(df[['trend_direction', 'content_age_days']].head(5))

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why simple IF-statements fail:**
1. **Volume & Context Sensitivity:** A 20% drop on a page losing 10,000 clicks is a critical business loss. A 20% drop on a page losing 2 clicks is statistical noise. Static percentage rules trigger false alarms on low-volume pages while missing subtle, high-impact declines.
2. **Non-linear Feature Interactions:** Traffic decline depends on combinations of signals-impressions, click-through rates (CTR), seasonal baselines, and page age. Hardcoded rules turn into an unmaintainable maze of edge cases.
3. **Dynamic Adaptation:** Fixed rules fail when sitewide seasonality occurs. ML scoring ranks pages relative to each other dynamically rather than relying on rigid static thresholds.

In [ ]:
import pandas as pd

# Demonstrating why a static 'drop > 20%' rule fails without traffic volume context
rule_demo = pd.DataFrame({
    "page_type": ["High-Traffic Core Page", "Low-Traffic Blog Post"],
    "prev_traffic": [50000, 10],
    "curr_traffic": [40000, 8],
    "pct_change": ["-20%", "-20%"],
    "absolute_click_loss": [10000, 2],
    "rule_flagged": [True, True]
})

print("Comparison showing static rule flaw:")
print(rule_demo[["page_type", "pct_change", "absolute_click_loss", "rule_flagged"]])

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.